# 实验3.1 昇腾平台 GPIO 驱动与 CANN Runtime 协同控制仿真实验

**副标题**：基于仿真环境的嵌入式 AI 外设控制与硬件资源调度实践

## 实验目的

本实验基于已预先调通并搭建好的嵌入式驱动与 CANN 协同开发基础环境，聚焦 GPIO 驱动与 AI 算力调用中的关键问题点进行设计。学生将在仿真与真实结合的昇腾平台上，重点实现：

- 利用 **libgpiod** 库完成 GPIO 外设控制（硬件/仿真双模式）
- 通过 **CANN Runtime** 接口调用 AI 算力（模拟推理流程）
- 实践**硬件资源调度与协同运行**（Stream 并行调度）
- 掌握昇腾平台驱动与 AI 框架联合开发、硬件资源管理及端侧部署的核心方法

> **设计理念**：本实验采用软硬件解耦设计，学生可以在纯软件仿真环境下理解完整流程，亦可直接部署到真实硬件上运行，降低硬件依赖门槛，聚焦核心知识掌握。

## 实验架构

```
┌──────────────────────────────────────────────────────────┐
│               实验3：GPIO + CANN 协同控制                │
├──────────────────────────────────────────────────────────┤
│  实验环境（双模式）                                      │
│  ┌───────────────────┐   ┌───────────────────────────┐  │
│  │  硬件模式         │   │  仿真模式（默认）         │  │
│  │  • 真实GPIO操作   │   │  • 虚拟GPIO状态机         │  │
│  │  • 真实NPU推理    │   │  • 模拟推理耗时           │  │
│  │  • 需要root权限   │   │  • 无需特殊权限           │  │
│  └───────────────────┘   └───────────────────────────┘  │
│  核心实验模块                                            │
│  ┌────────────┐  ┌────────────┐  ┌──────────────────┐  │
│  │ 模块1      │  │ 模块2      │  │ 模块3            │  │
│  │ GPIO外设   │  │ CANN资源   │  │ 协同控制与部署   │  │
│  │ (libgpiod) │  │ (ACL/Stream)│  │ (GPIO触发→AI推理)│ │
│  └────────────┘  └────────────┘  └──────────────────┘  │
└──────────────────────────────────────────────────────────┘
```

## 背景知识

在正式开始实验之前，我们需要了解昇腾平台的软件体系架构、嵌入式 Linux 系统的启动流程、Linux 驱动开发的基本方式以及 CANN 异构计算架构。这些知识将帮助我们理解本实验中 GPIO 控制与 AI 推理协同工作的底层原理。

### 0.1 昇腾软件体系架构

昇腾的软件体系是一个典型的、层层递进的全栈式架构，其设计哲学是**软硬件协同与分层解耦**。该体系自底向上可分为四个核心层次：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">层次</th>
<th style="text-align: left;">名称</th>
<th style="text-align: left;">主要功能</th>
</tr>
<tr>
<td style="text-align: left;">第一层</td>
<td style="text-align: left;"><strong>操作系统层</strong></td>
<td style="text-align: left;">以 Ubuntu/openEuler 为基础平台，内核集成昇腾驱动，管理硬件资源</td>
</tr>
<tr>
<td style="text-align: left;">第二层</td>
<td style="text-align: left;"><strong>CANN 异构计算架构</strong></td>
<td style="text-align: left;">连接上层生态与底层硬件的核心桥梁，含驱动层、运行时库、图编译器、AscendCL</td>
</tr>
<tr>
<td style="text-align: left;">第三层</td>
<td style="text-align: left;"><strong>框架与模型层</strong></td>
<td style="text-align: left;">MindSpore 原生框架 + PyTorch/TensorFlow 适配器，提供友好开发界面</td>
</tr>
<tr>
<td style="text-align: left;">第四层</td>
<td style="text-align: left;"><strong>应用与工具层</strong></td>
<td style="text-align: left;">行业解决方案、预训练模型库、MindStudio 集成开发环境</td>
</tr>
</table>

- **操作系统层**是整个体系的基石，其内核集成的昇腾驱动如同"翻译官"，将标准操作系统指令转换为硬件能理解的信号。
- **CANN** 是整个体系的心脏与大脑，图编译器通过算子融合、常量折叠等优化，将神经网络模型转换为高度优化的离线模型；AscendCL 提供面向开发者的底层 API。
- **框架与模型层**中，MindSpore 与昇腾硬件达到"血脉相通"的协同级别，同时通过适配器模式支持主流框架无缝融入。
- **应用与工具层**中，MindStudio 集成了模型转换、性能分析、应用调试等工具，是软硬件协同思想在开发流程中的集中体现。

本实验主要涉及**操作系统层**（GPIO 驱动）与 **CANN 层**（AscendCL Runtime 接口）的协同调用。

### 0.2 嵌入式 Linux 系统启动流程

一个完整的嵌入式 Linux 系统在逻辑上可划分为四个紧密协作的层次：**引导加载程序（BootLoader）、Linux 内核、文件系统、用户应用程序**。启动流程如下：

1. **BootLoader 阶段**：设备上电后第一个运行的软件（常用 U-Boot），执行底层硬件初始化（时钟、内存控制器、串口），从 Flash/eMMC 加载内核映像到内存，并传递启动参数。
2. **内核阶段**：内核接管系统软硬件资源，进行进程调度、内存管理、中断处理和设备驱动初始化，挂载根文件系统。
3. **文件系统阶段**：提供运行环境，包含系统程序、命令工具（如 BusyBox）、系统库和配置文件。
4. **用户应用阶段**：内核启动第一个用户空间进程（`/sbin/init`），最终运行用户应用程序。

![昇腾操作系统运行流程](images/ascend_os_boot_flow.png)

> **图示：昇腾操作系统运行流程**（从 BootLoader 硬件初始化 → 内核加载 → 文件系统挂载 → 用户应用启动的完整链路）

在本实验中，GPIO 驱动和 CANN Runtime 都运行在**用户应用阶段**，通过操作系统提供的接口访问底层硬件。

### 0.3 Linux 驱动开发的三种方式

Linux 系统中，驱动程序作为硬件与操作系统及应用软件之间的关键桥梁。面对不同需求，开发者可从三种方案中选择：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方案</th>
<th style="text-align: left;">运行空间</th>
<th style="text-align: left;">优势</th>
<th style="text-align: left;">劣势</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;"><strong>传统内核驱动</strong></td>
<td style="text-align: left;">内核态</td>
<td style="text-align: left;">性能最优，完整利用内核功能</td>
<td style="text-align: left;">开发复杂，调试困难</td>
<td style="text-align: left;">复杂加速器、高性能设备</td>
</tr>
<tr>
<td style="text-align: left;"><strong>纯用户空间驱动</strong></td>
<td style="text-align: left;">用户态</td>
<td style="text-align: left;">开发简单，调试友好，错误不崩溃系统</td>
<td style="text-align: left;">性能有瓶颈，功能受限</td>
<td style="text-align: left;">简单外设、快速原型验证</td>
</tr>
<tr>
<td style="text-align: left;"><strong>寄存器级驱动</strong></td>
<td style="text-align: left;">内核态</td>
<td style="text-align: left;">极致性能，完全控制</td>
<td style="text-align: left;">开发最难，可移植性差</td>
<td style="text-align: left;">特殊硬件、极致性能场景</td>
</tr>
</table>

本实验采用**纯用户空间驱动**方案，通过 libgpiod 库在用户空间控制 GPIO，兼顾了开发简便性与安全性，是教学和快速原型的首选方案。

#### libgpiod：现代 GPIO 用户空间控制

由于传统 Sysfs 接口（`/sys/class/gpio/`）存在竞态条件、不支持中断 polling 等缺陷，Linux GPIO 子系统推出了更先进的替代方案：

- **libgpiod**：官方推荐的现代 C 库和工具集，基于内核新 GPIO 字符设备接口（`/dev/gpiochipX`），提供 `gpiod_chip_open`、`gpiod_line_request_output` 等安全 API 和 `gpioset`、`gpioget` 等命令行工具。
- **操作流程**：`Chip('gpiochip0')` → `get_line(pin)` → `request(consumer, type)` → `set_value/get_value` → `release()`

本实验的 `GPIOController` 类在硬件模式下正是通过 libgpiod 的上述流程操作真实 GPIO 引脚。

### 0.4 设备树机制简介

设备树（Device Tree）是嵌入式世界的"硬件说明书"，让 Linux 内核不再靠"硬编码"去猜硬件长什么样，而是直接读取一份清晰的配置文件。

**核心思想**：把硬件拓扑结构从内核代码里剥离出来，变成独立的数据结构，在启动时由 BootLoader 加载并传递给内核。同一个内核镜像就能适配多种不同的硬件布局，实现"一次编译，到处运行"。

设备树采用类 C 风格的文本格式 `.dts`，经编译后生成二进制 `.dtb` 文件。其三大基石为：

- **节点（node）**：`{}` 包裹的内容，代表一个物理或逻辑设备，如 `serial@1c28000` 表示地址为 `0x1c28000` 的串口控制器。
- **属性（property）**：键值对，最关键的是 `compatible` 属性——驱动匹配的"身份证"，内核拿着它遍历 `of_match_table`，命中即调用 `.probe()` 函数。
- **标签（label）**：如 `uart0: serial@1c28000 {...}`，其他地方可用 `<&uart0>` 引用，避免重复书写长路径。

```dts
// 设备树节点示例：UART 控制器
uart0: serial@1c28000 {
    compatible = "snps,dw-apb-uart";   // 驱动匹配字符串
    reg = <0x1c28000 0x1000>;          // 寄存器地址和大小
    interrupts = <0 32 4>;             // 中断号
    clocks = <&clk_uart0>;             // 时钟引用
    status = "okay";                   // 启用状态
};
```

> **与本实验的关系**：在真实昇腾硬件上，GPIO 引脚的编号、地址和中断等信息正是通过设备树描述并传递给内核的。仿真模式下我们用 Python 字典模拟了这一配置过程。

### 0.5 CANN 异构计算架构

CANN（Compute Architecture for Neural Networks）是昇腾连接上层生态与底层硬件的核心桥梁。其关键组件包括：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">组件</th>
<th style="text-align: left;">功能</th>
<th style="text-align: left;">本实验对应</th>
</tr>
<tr>
<td style="text-align: left;"><strong>驱动层</strong></td>
<td style="text-align: left;">直接管理硬件设备</td>
<td style="text-align: left;">GPIO 驱动（libgpiod）</td>
</tr>
<tr>
<td style="text-align: left;"><strong>运行时库（Runtime）</strong></td>
<td style="text-align: left;">上下衔接，管理 Device/Context/Stream</td>
<td style="text-align: left;"><code>CANNRuntime</code> 类</td>
</tr>
<tr>
<td style="text-align: left;"><strong>任务调度（TSDB）</strong></td>
<td style="text-align: left;">芯片内部任务调度的"交通指挥官"</td>
<td style="text-align: left;">多 Stream 并行调度</td>
</tr>
<tr>
<td style="text-align: left;"><strong>图编译器</strong></td>
<td style="text-align: left;">算子融合、常量折叠等优化</td>
<td style="text-align: left;">模拟推理中的矩阵乘+ReLU</td>
</tr>
<tr>
<td style="text-align: left;"><strong>AscendCL</strong></td>
<td style="text-align: left;">面向开发者的底层 API</td>
<td style="text-align: left;"><code>acl.rt.*</code> 接口封装</td>
</tr>
</table>

#### Host-Device 异构架构关键概念

- **Device**：昇腾 AI 处理器（NPU），执行 AI 计算任务。通过 `acl.rt.set_device()` 设置当前使用的设备。
- **Context**：执行上下文，封装设备资源和执行环境，是 Stream 和内存操作的前提。
- **Stream**：异步执行队列。同一 Stream 内任务**串行**执行，不同 Stream 间可**并行**。本实验创建多个 Stream 演示并行调度。
- **H2D / D2H**：Host→Device / Device→Host 数据传输，分别是将输入数据送到 NPU 和将推理结果取回主机。

```
Host (CPU)                          Device (NPU)
┌──────────────┐    H2D传输       ┌──────────────┐
│  输入数据     │  ──────────→   │  Device内存   │
│  (numpy数组)  │                │  推理执行     │
│  结果接收     │  ←──────────   │  (Stream调度) │
└──────────────┘    D2H传输       └──────────────┘
```

理解这些概念是掌握本实验模块2（CANN Runtime 资源调度）和模块3（协同控制）的关键。

## 第一部分：实验准备与环境配置

### 1.1 安装依赖

#### 知识点说明

本代码块用于检查并安装实验所需的 Python 依赖包。在嵌入式 AI 开发中，环境配置是第一步也是最容易出错的环节。我们需要理解：

- **numpy**：Python 数值计算库，用于数组操作和矩阵运算。在本实验中，AI 推理的输入/输出数据都以 numpy 数组形式在 Host 端表示。
- **gpiod**：Linux GPIO 字符设备的 Python 绑定，对应背景知识中介绍的 libgpiod 库。仅在**硬件模式**下需要（直接操作真实 GPIO 引脚）；**仿真模式**下不需要，安装失败可忽略。
- **依赖检测策略**：先尝试 `__import__` 导入，已安装则跳过；未安装则通过 `subprocess` 调用 `pip install` 安装。这种"按需安装"模式在 Jupyter Notebook 中非常常见。

#### 预期结果

numpy 通常已预装，显示 `[OK] numpy 已安装`；gpiod 在仿真模式下不需要，安装失败可忽略。在昇腾云沙箱环境中，numpy 通常已可用。

In [ ]:
# 在Jupyter Notebook中安装依赖
import subprocess
import sys

def install_dependencies():
    """安装实验所需依赖"""
    packages = [
        'numpy',
        'gpiod',           # GPIO控制（可选，仿真模式下不需要）
    ]
    for pkg in packages:
        try:
            __import__(pkg.replace('-', '_'))
            print(f"[OK] {pkg} 已安装")
        except ImportError:
            print(f"正在安装 {pkg}...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
            except Exception as e:
                print(f"[!] {pkg} 安装失败（仿真模式可忽略）: {e}")

install_dependencies()

### 1.2 环境检测

#### 知识点说明

本代码块检测 Python 运行环境与 CANN 可用性，并自动决定使用**真实模式**还是**仿真模式**。这是软硬件解耦设计的关键环节：

- **导入基础库**：`os`/`sys`/`time`/`threading` 用于系统操作和并发控制；`numpy` 用于数值计算；`Enum`/`dataclass`/`typing` 用于类型定义；`warnings` 用于屏蔽无关警告。
- **CANN 检测逻辑**：尝试 `import acl` 并调用 `acl.init()`，返回值为 0 表示 CANN 环境可用（真实模式）；若导入失败或初始化异常，则自动回退到仿真模式。
- **双模式意义**：仿真模式让无昇腾硬件的学习者也能完整运行实验流程；真实模式则在昇腾设备上调用真实 NPU 算力。两者接口一致，切换零成本。

#### 预期结果

输出 Python 版本和运行环境（Jupyter Notebook）。在昇腾 CANN 环境中显示 `[OK] CANN环境可用（真实模式）`；在非昇腾环境中显示 `[!] CANN未安装，将使用仿真模式`。仿真模式是本实验的默认模式。

In [ ]:
import os
import sys
import time
import threading
import numpy as np
from enum import Enum
from dataclasses import dataclass
from typing import Optional, List, Tuple, Callable
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("实验3：GPIO + CANN 协同控制实验")
print("=" * 60)
print(f"Python版本: {sys.version.split()[0]}")
print(f"运行环境: {'Jupyter Notebook' if 'ipykernel' in sys.modules else 'Python Script'}")

# 检测CANN是否可用
CANN_AVAILABLE = False
try:
    import acl
    ret = acl.init()
    if ret == 0:
        CANN_AVAILABLE = True
        acl.finalize()
        print("[OK] CANN环境可用（真实模式）")
    else:
        print("[!] CANN初始化失败，将使用仿真模式")
except ImportError:
    print("[!] CANN未安装，将使用仿真模式")
except Exception:
    print("[!] CANN环境异常，将使用仿真模式")

## 第二部分：模块1 - GPIO 外设控制

### 2.1 GPIO 控制器抽象（硬件/仿真双模式）

#### 知识点说明

本代码块是 GPIO 控制模块的核心，定义了 `GPIOMode` 枚举、`GPIOEvent` 数据类和 `GPIOController` 控制器类。这是面向对象设计中**策略模式**的典型应用——通过 `mode` 参数在仿真和硬件模式间切换，对外接口完全一致。

**类设计解析**：

1. **`GPIOMode` 枚举**：定义 `SIMULATION`（仿真）和 `HARDWARE`（硬件）两种工作模式，使模式选择清晰类型安全。

2. **`GPIOEvent` 数据类**：使用 `@dataclass` 装饰器自动生成构造函数，记录每次 GPIO 事件的引脚号、电平值、时间戳和描述。这是"事件溯源"模式的简化实现——所有操作都被记录，便于后续分析和调试。

3. **`GPIOController` 控制器类**核心方法：
   - **`__init__`**：根据 mode 调用 `_init_simulation` 或 `_init_hardware`。仿真模式在内存字典 `_pins` 中初始化 6 个虚拟引脚（17/18/22/23/24/25）；硬件模式通过 `gpiod.Chip('gpiochip0')` 打开 GPIO 芯片。
   - **`write(pin, value)`**：写入引脚电平。硬件模式下走 libgpiod 完整流程：`get_line` → `request(DIR_OUT)` → `set_value` → `release`；仿真模式下直接修改字典值并打印。写入后记录事件并触发回调。
   - **`read(pin)`**：读取引脚电平。仿真模式下附带 5% 随机抖动模拟真实信号的不稳定性，让仿真更贴近现实。
   - **`on_event(pin, callback)`** / **`_trigger_callbacks`**：事件回调机制，允许外部代码注册引脚变化监听器。这是实现"GPIO 触发→AI 推理"协同控制的基础——当引脚电平变化时，自动通知注册的回调函数。
   - **`simulate_trigger(pin, duration)`**：仿真模式专用，产生一个持续 `duration` 秒的高电平脉冲，模拟按键按下或传感器触发信号。

**硬件模式 libgpiod 调用链**：
```
gpiod.Chip('gpiochip0')  →  chip.get_line(pin)  →  line.request(consumer, type=DIR_OUT/DIR_IN)
                          →  line.set_value(v) / line.get_value()  →  line.release()
```

#### 预期结果

执行后输出 `[GPIO] 仿真模式已启用`，表示 GPIOController 已在仿真模式下初始化完成，6 个虚拟引脚就绪。

In [ ]:
class GPIOMode(Enum):
    """GPIO工作模式"""
    SIMULATION = "simulation"  # 仿真模式（默认）
    HARDWARE = "hardware"      # 硬件模式


@dataclass
class GPIOEvent:
    """GPIO事件"""
    pin: int
    value: int
    timestamp: float
    description: str


class GPIOController:
    """
    GPIO控制器（支持硬件和仿真双模式）

    设计理念：
    - 抽象GPIO操作接口，硬件/仿真模式无缝切换
    - 仿真模式无需真实硬件，适合教学
    - 硬件模式直接操作真实GPIO
    """

    def __init__(self, mode: GPIOMode = GPIOMode.SIMULATION):
        self.mode = mode
        self._pins = {}       # 存储引脚状态
        self._events = []     # 事件记录
        self._callbacks = {}  # 回调函数

        if mode == GPIOMode.HARDWARE:
            self._init_hardware()
        else:
            self._init_simulation()

    def _init_simulation(self):
        """初始化仿真模式"""
        print("[GPIO] 仿真模式已启用")
        for pin in [17, 18, 22, 23, 24, 25]:
            self._pins[pin] = 0

    def _init_hardware(self):
        """初始化硬件模式"""
        try:
            import gpiod
            self._gpiod = gpiod
            self._chip = gpiod.Chip('gpiochip0')
            print("[GPIO] 硬件模式初始化成功")
        except ImportError:
            print("[GPIO] gpiod未安装，切换到仿真模式")
            self.mode = GPIOMode.SIMULATION
            self._init_simulation()
        except Exception as e:
            print(f"[GPIO] 硬件初始化失败: {e}，切换到仿真模式")
            self.mode = GPIOMode.SIMULATION
            self._init_simulation()

    def set_mode(self, pin: int, direction: str):
        """设置引脚方向 (in/out)"""
        if pin not in self._pins and self.mode == GPIOMode.SIMULATION:
            self._pins[pin] = 0
        return True

    def write(self, pin: int, value: int):
        """写入GPIO值"""
        if self.mode == GPIOMode.HARDWARE:
            try:
                line = self._chip.get_line(pin)
                line.request(consumer="gpio_control",
                             type=self._gpiod.LINE_REQ_DIR_OUT)
                line.set_value(value)
                line.release()
            except Exception as e:
                print(f"[GPIO] 写入失败: {e}")
                return False
        else:
            self._pins[pin] = value
            print(f"[GPIO仿真] GPIO{pin} = {value}")

        self._events.append(GPIOEvent(
            pin=pin, value=value,
            timestamp=time.time(),
            description=f"Write to GPIO{pin}"
        ))
        self._trigger_callbacks(pin, value)
        return True

    def read(self, pin: int) -> int:
        """读取GPIO值"""
        if self.mode == GPIOMode.HARDWARE:
            try:
                line = self._chip.get_line(pin)
                line.request(consumer="gpio_control",
                             type=self._gpiod.LINE_REQ_DIR_IN)
                value = line.get_value()
                line.release()
                return value
            except Exception as e:
                print(f"[GPIO] 读取失败: {e}")
                return 0
        else:
            value = self._pins.get(pin, 0)
            # 模拟信号变化（随机抖动，让仿真更真实）
            if np.random.random() < 0.05:  # 5%的概率变化
                value = 1 - value
                self._pins[pin] = value
            return value

    def on_event(self, pin: int, callback: Callable):
        """注册事件回调"""
        if pin not in self._callbacks:
            self._callbacks[pin] = []
        self._callbacks[pin].append(callback)

    def _trigger_callbacks(self, pin: int, value: int):
        """触发回调函数"""
        if pin in self._callbacks:
            for cb in self._callbacks[pin]:
                try:
                    cb(pin, value)
                except Exception as e:
                    print(f"[GPIO] 回调执行失败: {e}")

    def get_events(self) -> List[GPIOEvent]:
        """获取事件历史"""
        return self._events

    def simulate_trigger(self, pin: int, duration: float = 0.5):
        """模拟触发信号（仿真模式专用）"""
        if self.mode != GPIOMode.SIMULATION:
            return
        print(f"[GPIO仿真] 模拟触发 GPIO{pin} (持续{duration}s)")
        self.write(pin, 1)
        time.sleep(duration)
        self.write(pin, 0)

    def cleanup(self):
        """清理资源"""
        print("[GPIO] 资源清理完成")

### 2.2 GPIO 基础操作演示

#### 知识点说明

本代码块通过 `gpio_basic_demo()` 函数演示 GPIO 控制器的四类基础操作，帮助学生从实践角度理解 GPIO 控制原理：

1. **写入操作（LED 闪烁模拟）**：对 GPIO17/18/22 依次写 1（高电平，模拟 LED 亮）再写 0（低电平，模拟 LED 灭），每次操作间隔 0.2/0.1 秒。在真实硬件上，这对应 LED 的闪烁效果；仿真模式下通过打印 `[GPIO仿真] GPIO17 = 1` 展示状态变化。

2. **读取操作（电平检测）**：连续 5 次读取 GPIO18 的电平值。由于仿真模式 `read()` 方法内置 5% 随机抖动，偶尔会出现电平翻转，模拟真实传感器信号的不稳定性。每次读取输出电平值和中文状态描述（高电平/低电平）。

3. **模拟触发信号（脉冲生成）**：调用 `simulate_trigger(18, duration=0.8)` 在 GPIO18 上产生持续 0.8 秒的高电平脉冲。这模拟了按键按下或传感器触发的场景——先写 1，等待 0.8 秒，再写 0。

4. **事件记录查看**：调用 `get_events()` 获取最近 5 条事件记录，显示操作描述和时间戳。这展示了 GPIOEvent 事件溯源机制——所有操作都被完整记录，可用于审计和调试。

#### 预期结果

- 写入操作：输出 `[GPIO仿真] GPIO17 = 1`、`GPIO17 = 0` 等共 6 条写入信息
- 读取操作：输出 5 次 GPIO18 电平值（偶尔因 5% 抖动出现翻转）
- 模拟触发：输出先 1 后 0 的电平变化，间隔 0.8 秒
- 事件记录：显示最近 5 次操作的描述和时间戳

In [ ]:
def gpio_basic_demo():
    """GPIO基础操作演示"""
    print("\n" + "=" * 60)
    print("[模块1] GPIO外设控制演示")
    print("=" * 60)

    gpio = GPIOController(mode=GPIOMode.SIMULATION)
    print(f"GPIO模式: {gpio.mode.value}")

    # 1. 写入操作
    print("\n[1] GPIO写入操作")
    print("-" * 40)
    for pin in [17, 18, 22]:
        gpio.write(pin, 1)
        time.sleep(0.2)
        gpio.write(pin, 0)
        time.sleep(0.1)

    # 2. 读取操作
    print("\n[2] GPIO读取操作")
    print("-" * 40)
    for i in range(5):
        value = gpio.read(18)
        status = "高电平" if value == 1 else "低电平"
        print(f"  第{i+1}次: GPIO18 = {value} ({status})")
        time.sleep(0.3)

    # 3. 模拟触发
    print("\n[3] 模拟触发信号")
    print("-" * 40)
    gpio.simulate_trigger(18, duration=0.8)

    # 4. 事件记录
    print("\n[4] 事件记录")
    print("-" * 40)
    events = gpio.get_events()[-5:]
    for event in events:
        print(f"  {event.description} @ {event.timestamp:.2f}")

    gpio.cleanup()
    return gpio

gpio_controller = gpio_basic_demo()

## 第三部分：模块2 - CANN Runtime 资源调度

### 3.1 CANN Runtime 封装（支持仿真模式）

#### 知识点说明

本代码块定义了 `CANNRuntime` 类，封装昇腾 ACL（AscendCL）的**五大核心功能**。这是理解异构计算架构的关键代码，每个功能都对应背景知识中介绍的 CANN 概念：

**五大功能详解**：

1. **设备管理（Device）**：`_init_hardware` 中通过 `acl.rt.set_device(device_id)` 设置当前使用的 NPU 设备。多卡场景下可切换不同设备（device_id=0,1,2...）。仿真模式下用字符串标记模式。

2. **上下文管理（Context）**：通过 `acl.rt.create_context(device_id)` 创建执行上下文。Context 封装了设备资源和执行环境，是 Stream 和内存操作的前提。仿真模式下 Context 为 None（不需要真实上下文）。

3. **Stream 调度**：`create_stream()` 创建异步执行队列。同一 Stream 内任务串行执行，不同 Stream 间可并行。本实验初始化时创建 2 个 Stream，演示并行调度。仿真模式下用 `sim_stream_0`、`sim_stream_1` 等字符串标识。

4. **内存管理（Memory）**：
   - `allocate_memory(size)`：硬件模式下调用 `acl.rt.malloc()` 在 Device 端分配内存；仿真模式下用虚拟地址 `id(self) + len(device_memory)` 模拟。
   - `memcpy_h2d(host_data, dev_mem)`：Host→Device 数据传输，将 numpy 数组发送到 NPU。统计 `h2d_transfers` 次数。
   - `memcpy_d2h(dev_mem, size)`：Device→Host 数据传输，将推理结果取回主机。统计 `d2h_transfers` 次数。

5. **推理执行（Inference）**：`execute_inference(input_data, stream)` 执行 AI 推理。
   - 仿真模式 `_simulate_inference`：用 `矩阵乘法 + ReLU 激活` 模拟一层神经网络的前向传播，并按数据规模 `sleep` 模拟计算延迟（`0.1 + data.size/100000` 秒，上限 0.5 秒）。
   - 硬件模式 `_hardware_inference`：接入真实模型时可替换为 AscendCL 推理调用。

**统计信息 `stats`**：记录 `h2d_transfers`（H2D 传输次数）、`d2h_transfers`（D2H 传输次数）、`inference_calls`（推理调用次数）、`total_time`（总耗时），用于性能分析。

**自动模式选择**：`mode="auto"` 时尝试导入 `acl` 并初始化，成功则用硬件模式，失败则回退仿真模式。

#### 预期结果

执行后输出 `[CANN] Runtime模式: simulation` 和 `[CANN仿真] 初始化完成`，表示 CANNRuntime 在仿真模式下初始化成功，2 个虚拟 Stream 已创建。

In [ ]:
class CANNRuntime:
    """
    CANN Runtime封装（支持硬件和仿真双模式）

    核心功能：
    1. 设备管理（Device）
    2. 上下文管理（Context）
    3. Stream调度（Stream）
    4. 内存管理（Memory）
    5. 推理执行（Inference）
    """

    def __init__(self, device_id: int = 0, mode: str = "auto"):
        self.device_id = device_id
        self.mode = mode
        self.initialized = False
        self.context = None
        self.streams = []
        self.device_memory = []
        self.stats = {
            'h2d_transfers': 0,
            'd2h_transfers': 0,
            'inference_calls': 0,
            'total_time': 0
        }
        self._init_runtime()

    def _init_runtime(self):
        """初始化Runtime"""
        if self.mode == "auto":
            try:
                import acl
                ret = acl.init()
                if ret == 0:
                    self.mode = "hardware"
                    acl.finalize()
                else:
                    self.mode = "simulation"
            except Exception:
                self.mode = "simulation"

        print(f"[CANN] Runtime模式: {self.mode}")
        if self.mode == "hardware":
            self._init_hardware()
        else:
            self._init_simulation()

    def _init_simulation(self):
        """仿真模式初始化"""
        print("[CANN仿真] 初始化完成")
        self.initialized = True
        for i in range(2):
            self.streams.append(f"sim_stream_{i}")

    def _init_hardware(self):
        """硬件模式初始化"""
        try:
            import acl
            self._acl = acl
            ret = acl.init()
            if ret != 0:
                raise RuntimeError(f"ACL初始化失败: {ret}")
            ret = acl.rt.set_device(self.device_id)
            if ret != 0:
                raise RuntimeError(f"设置设备失败: {ret}")
            self.context = acl.rt.create_context(self.device_id)
            for i in range(2):
                stream = acl.rt.create_stream()
                self.streams.append(stream)
            self.initialized = True
            print(f"[CANN硬件] 初始化成功，设备{self.device_id}")
        except Exception as e:
            print(f"[CANN硬件] 初始化失败: {e}，切换到仿真模式")
            self.mode = "simulation"
            self.streams = []
            self._init_simulation()

    def create_stream(self):
        """创建Stream"""
        if self.mode == "hardware":
            try:
                stream = self._acl.rt.create_stream()
                self.streams.append(stream)
                return stream
            except Exception as e:
                print(f"[CANN] 创建Stream失败: {e}")
                return None
        else:
            stream_id = f"sim_stream_{len(self.streams)}"
            self.streams.append(stream_id)
            print(f"[CANN仿真] 创建Stream: {stream_id}")
            return stream_id

    def allocate_memory(self, size: int):
        """分配Device内存"""
        if self.mode == "hardware":
            try:
                mem = self._acl.rt.malloc(size, 0)
                self.device_memory.append(mem)
                return mem
            except Exception as e:
                print(f"[CANN] 内存分配失败: {e}")
                return None
        else:
            mem_addr = id(self) + len(self.device_memory)
            self.device_memory.append(mem_addr)
            print(f"[CANN仿真] 分配内存: {size}字节 -> 地址{hex(mem_addr)}")
            return mem_addr

    def memcpy_h2d(self, host_data: np.ndarray, dev_mem):
        """Host到Device数据传输"""
        self.stats['h2d_transfers'] += 1
        size = host_data.nbytes
        if self.mode == "hardware":
            try:
                self._acl.rt.memcpy(
                    host_data.tobytes(), dev_mem, size,
                    self._acl.rt.MEMCPY_HOST_TO_DEVICE
                )
                print(f"[CANN] H2D拷贝完成: {size}字节")
            except Exception as e:
                print(f"[CANN] H2D拷贝失败: {e}")
        else:
            print(f"[CANN仿真] H2D拷贝: {size}字节")
            self._last_h2d_data = host_data.copy()

    def memcpy_d2h(self, dev_mem, size: int) -> np.ndarray:
        """Device到Host数据传输"""
        self.stats['d2h_transfers'] += 1
        if self.mode == "hardware":
            try:
                data = self._acl.rt.memcpy(
                    dev_mem, size,
                    self._acl.rt.MEMCPY_DEVICE_TO_HOST
                )
                result = np.frombuffer(data, dtype=np.float32)
                print(f"[CANN] D2H拷贝完成: {size}字节")
                return result
            except Exception as e:
                print(f"[CANN] D2H拷贝失败: {e}")
                return np.zeros(size // 4, dtype=np.float32)
        else:
            print(f"[CANN仿真] D2H拷贝: {size}字节")
            return np.random.randn(size // 4).astype(np.float32)

    def execute_inference(self, input_data: np.ndarray, stream=None) -> np.ndarray:
        """执行推理（模拟/真实）"""
        self.stats['inference_calls'] += 1
        start_time = time.time()
        print(f"[推理] 开始执行 (输入形状: {input_data.shape})")
        if self.mode == "hardware":
            result = self._hardware_inference(input_data, stream)
        else:
            result = self._simulate_inference(input_data)
        elapsed = time.time() - start_time
        self.stats['total_time'] += elapsed
        print(f"[推理] 完成，耗时: {elapsed:.4f}秒")
        return result

    def _simulate_inference(self, input_data: np.ndarray) -> np.ndarray:
        """模拟推理（教学核心）"""
        data = input_data.reshape(1, -1) if input_data.ndim == 1 else input_data
        weight = np.random.randn(data.shape[-1], data.shape[-1]) * 0.1
        output = data @ weight                       # 线性变换
        output = np.maximum(output, 0)               # ReLU激活
        delay = min(0.1 + data.size / 100000, 0.5)   # 模拟计算延迟
        time.sleep(delay)
        return output.astype(np.float32)

    def _hardware_inference(self, input_data: np.ndarray, stream) -> np.ndarray:
        """真实硬件推理（接入真实模型时可替换实现）"""
        return self._simulate_inference(input_data)

    def synchronize(self):
        """同步所有Stream"""
        if self.mode == "hardware":
            try:
                self._acl.rt.synchronize_device()
            except Exception as e:
                print(f"[CANN] 同步失败: {e}")
        else:
            print("[CANN仿真] 同步完成")

    def get_stats(self) -> dict:
        """获取统计信息"""
        return self.stats

    def cleanup(self):
        """清理资源"""
        if self.mode == "hardware":
            try:
                for stream in self.streams:
                    if stream:
                        self._acl.rt.destroy_stream(stream)
                if self.context:
                    self._acl.rt.destroy_context(self.context)
                for mem in self.device_memory:
                    if mem:
                        self._acl.rt.free(mem)
                self._acl.rt.reset_device(self.device_id)
                self._acl.finalize()
            except Exception as e:
                print(f"[CANN] 清理失败: {e}")
        else:
            print("[CANN仿真] 资源清理完成")
        print("[CANN] 清理完成")

### 3.2 CANN Runtime 演示

#### 知识点说明

本代码块通过 `cann_runtime_demo()` 函数完整演示 CANN Runtime 的四大功能，帮助学生理解 Host-Device 异构架构的数据流：

1. **内存管理演示**：调用 `allocate_memory(1024 * 4)` 在 Device 端分配 4096 字节内存（1024 个 float32）。仿真模式下返回虚拟地址并打印，硬件模式下返回真实 Device 内存指针。

2. **数据传输演示（H2D / D2H）**：
   - 生成 1024 个随机 float32 数组作为 Host 数据
   - `memcpy_h2d`：将 Host 数据传输到 Device 内存（模拟将输入数据送到 NPU）
   - `memcpy_d2h`：将 Device 内存数据传输回 Host（模拟将推理结果取回 CPU）
   - 这展示了 Host↔Device 的完整数据往返流程

3. **多 Stream 并行执行**：
   - 创建 2 个 Stream（`stream1`、`stream2`）
   - 使用 Python `threading.Thread` 在 2 个线程中分别提交 128×128 矩阵的推理任务
   - 两个线程**并行**执行，交替输出推理开始和完成信息
   - 这演示了"不同 Stream 间可并行"的核心概念——在真实 NPU 上，不同 Stream 的任务会被硬件调度器并行执行

4. **执行统计**：输出 `stats` 字典，显示 H2D 传输次数、D2H 传输次数、推理调用次数和总耗时。这些指标是性能调优的重要依据。

#### 预期结果

- 内存分配：输出虚拟地址（如 `0x...`）
- H2D/D2H 拷贝：输出字节数（4096 字节）
- 多 Stream 并行：两个线程交替输出推理开始和完成信息
- 统计信息：`h2d_transfers=1, d2h_transfers=1, inference_calls=2`

In [ ]:
def cann_runtime_demo():
    """CANN Runtime功能演示"""
    print("\n" + "=" * 60)
    print("[模块2] CANN Runtime资源调度演示")
    print("=" * 60)

    runtime = CANNRuntime(mode="simulation")

    # 1. 内存管理
    print("\n[1] 内存管理")
    print("-" * 40)
    data_size = 1024
    dev_mem = runtime.allocate_memory(data_size * 4)
    print(f"  Device内存地址: {dev_mem}")

    # 2. 数据传输
    print("\n[2] 数据传输 (H2D/D2H)")
    print("-" * 40)
    host_data = np.random.randn(data_size).astype(np.float32)
    print(f"  Host数据: {host_data[:5]}...")
    runtime.memcpy_h2d(host_data, dev_mem)
    result_data = runtime.memcpy_d2h(dev_mem, data_size * 4)
    print(f"  回传数据: {result_data[:5]}...")

    # 3. 多Stream并行
    print("\n[3] 多Stream并行执行")
    print("-" * 40)
    stream1 = runtime.create_stream()
    stream2 = runtime.create_stream()

    def run_task(stream_name, data):
        print(f"  [{stream_name}] 开始执行")
        result = runtime.execute_inference(data)
        print(f"  [{stream_name}] 完成: {result.shape}")

    threads = []
    for i, stream in enumerate([stream1, stream2]):
        data = np.random.randn(128, 128).astype(np.float32)
        t = threading.Thread(target=run_task, args=(f"Stream_{i}", data))
        threads.append(t)
        t.start()
    for t in threads:
        t.join()

    # 4. 统计信息
    print("\n[4] 执行统计")
    print("-" * 40)
    stats = runtime.get_stats()
    for key, value in stats.items():
        print(f"  {key}: {value}")

    runtime.cleanup()
    return runtime

cann_runtime = cann_runtime_demo()

## 第四部分：模块3 - GPIO 与 CANN 协同控制

### 4.1 协同控制器

#### 知识点说明

本代码块定义了 `GPIOCANNCoordinator` 类，是本实验的**核心**——将 GPIO 控制器和 CANN Runtime 连接起来，实现"外设触发→AI 推理"的完整端侧部署流程。这种设计模式在端侧 AI 部署中非常典型：传感器信号触发推理，推理结果驱动执行器。

**核心设计**：`GPIO 触发 → 任务队列 → AI 推理 → 结果输出`

**工作流程详解**：

1. **初始化（`__init__`）**：接收 GPIOController 和 CANNRuntime 实例，在触发引脚（默认 GPIO18）上注册事件回调 `self._on_gpio_trigger`。初始化任务队列 `task_queue`、结果队列 `result_queue` 和任务计数器。

2. **GPIO 触发回调（`_on_gpio_trigger`）**：当 GPIO 检测到高电平（`value == 1`）且系统运行中时被触发：
   - 递增任务计数器，分配唯一任务 ID
   - 生成模拟输入数据（1×256 随机向量，模拟传感器采集的特征向量）
   - 将任务添加到任务队列（含 ID、数据、时间戳）
   - 调用 `_process_task` 处理该任务

3. **任务处理（`_process_task`）**：完整模拟一次端侧 AI 推理流程：
   - **Step 1**：分配 Device 内存（`allocate_memory`）
   - **Step 2**：H2D 传输（`memcpy_h2d`，将输入数据送到 NPU）
   - **Step 3**：创建 Stream 并执行推理（`create_stream` + `execute_inference`）
   - **Step 4**：保存推理结果到结果队列，输出结果摘要（前 5 个元素）

4. **启动与仿真触发（`start` / `_simulate_triggers`）**：
   - `start()` 设置 `running = True`，开始监控 GPIO 触发引脚
   - 仿真模式下 `_simulate_triggers()` 自动模拟 3 次触发信号（每次间隔 1.5 秒），无需手动操作
   - 硬件模式下持续监控真实 GPIO 引脚电平变化

5. **停止与统计（`stop` / `_show_statistics`）**：输出总任务数、已完成数、待处理数和 CANN Runtime 统计信息。

**端侧部署模式**：
```
传感器 → GPIO高电平 → 回调触发 → 生成输入数据 → H2D传输 → NPU推理 → 结果输出 → 驱动执行器
```

#### 预期结果

执行后输出 `[协调器] 初始化完成，触发引脚: GPIO18`，表示协同控制器已就绪，等待 GPIO 触发。

In [ ]:
class GPIOCANNCoordinator:
    """
    GPIO + CANN 协同控制器

    核心设计：
    1. GPIO触发 → 任务队列 → AI推理 → 结果输出
    2. 多Stream并行处理多个触发事件
    3. 完整的端侧部署模式
    """

    def __init__(self, gpio_controller: GPIOController,
                 runtime: CANNRuntime,
                 trigger_pin: int = 18):
        self.gpio = gpio_controller
        self.runtime = runtime
        self.trigger_pin = trigger_pin
        self.running = False
        self.task_queue = []
        self.result_queue = []
        self.task_counter = 0

        # 注册GPIO事件回调
        self.gpio.on_event(trigger_pin, self._on_gpio_trigger)
        print(f"[协调器] 初始化完成，触发引脚: GPIO{trigger_pin}")

    def _on_gpio_trigger(self, pin: int, value: int):
        """GPIO触发回调"""
        if value == 1 and self.running:
            self.task_counter += 1
            task_id = self.task_counter
            print(f"\n[触发] 检测到GPIO{pin}高电平! 任务ID: {task_id}")

            # 生成模拟输入数据
            input_data = np.random.randn(1, 256).astype(np.float32)

            # 添加到任务队列
            self.task_queue.append({
                'id': task_id,
                'data': input_data,
                'timestamp': time.time()
            })

            # 处理任务
            self._process_task(task_id, input_data)

    def _process_task(self, task_id: int, input_data: np.ndarray):
        """处理AI推理任务"""
        print(f"[任务 {task_id}] 开始处理...")

        # 1. 分配Device内存（模拟）
        data_size = input_data.nbytes
        dev_mem = self.runtime.allocate_memory(data_size)

        # 2. H2D传输
        self.runtime.memcpy_h2d(input_data, dev_mem)

        # 3. 执行推理（使用Stream并行）
        stream = self.runtime.create_stream()
        result = self.runtime.execute_inference(input_data, stream)

        # 4. 获取结果
        self.result_queue.append({
            'id': task_id,
            'result': result,
            'timestamp': time.time()
        })

        print(f"[任务 {task_id}] 完成! 结果形状: {result.shape}")
        result_summary = result.flatten()[:5]
        print(f"  结果摘要: {result_summary}")

    def start(self):
        """启动协同控制系统"""
        print("\n" + "=" * 60)
        print("[协调器] 启动协同控制系统")
        print("=" * 60)
        print(f"监控GPIO{self.trigger_pin} (高电平触发)")
        print("按Ctrl+C停止监控")
        print("-" * 60)

        self.running = True

        # 如果是仿真模式，自动模拟触发
        if self.gpio.mode == GPIOMode.SIMULATION:
            self._simulate_triggers()

    def _simulate_triggers(self):
        """仿真模式：自动模拟触发信号"""
        print("[仿真] 开始模拟触发信号...")
        for i in range(3):
            print(f"\n[仿真] 第{i+1}次触发")
            self.gpio.simulate_trigger(self.trigger_pin, duration=0.5)
            time.sleep(1.5)
        self.stop()

    def stop(self):
        """停止协同控制系统"""
        self.running = False
        print("\n[协调器] 停止协同控制")
        self._show_statistics()

    def _show_statistics(self):
        """显示统计信息"""
        print("\n" + "=" * 60)
        print("[协调器] 执行统计")
        print("=" * 60)
        print(f"总任务数: {self.task_counter}")
        print(f"已完成: {len(self.result_queue)}")
        print(f"待处理: {len(self.task_queue)}")

        stats = self.runtime.get_stats()
        print("\nCANN Runtime统计:")
        for key, value in stats.items():
            print(f"  {key}: {value}")

    def get_results(self) -> List[dict]:
        """获取所有推理结果"""
        return self.result_queue

    def cleanup(self):
        """清理资源"""
        self.gpio.cleanup()
        self.runtime.cleanup()
        print("[协调器] 资源清理完成")

### 4.2 协同控制完整演示

#### 知识点说明

本代码块通过 `coordination_demo()` 函数演示 GPIO+CANN 协同控制的完整流程，将前三个模块串联起来：

**执行流程**：

1. **创建 GPIO 控制器**：`GPIOController(mode=GPIOMode.SIMULATION)` —— 仿真模式，无需真实硬件
2. **创建 CANN Runtime**：`CANNRuntime(mode="simulation")` —— 仿真模式，模拟 NPU 推理
3. **创建协同控制器**：`GPIOCANNCoordinator(gpio, runtime, trigger_pin=18)` —— 以 GPIO18 为触发引脚
4. **启动协同控制**：`coordinator.start()` —— 仿真模式下自动触发 3 次
5. **获取推理结果**：`coordinator.get_results()` —— 取回所有推理结果
6. **清理资源**：`coordinator.cleanup()` —— 释放 GPIO 和 CANN Runtime 资源

**每次触发的完整链路**：
```
GPIO18 高电平 → _on_gpio_trigger 回调 → 生成 1×256 随机输入
→ allocate_memory 分配 Device 内存
→ memcpy_h2d 数据传输到 NPU
→ create_stream 创建执行队列
→ execute_inference 执行推理（矩阵乘+ReLU）
→ 结果保存到 result_queue
→ GPIO18 低电平（触发结束）
```

**资源清理的重要性**：在真实硬件上，未释放的 GPIO 引脚、Stream 和 Device 内存会导致资源泄漏，最终使后续操作失败。`cleanup()` 方法确保所有资源被正确释放。

#### 预期结果

启动后自动模拟 3 次 GPIO18 高电平触发，每次触发产生一个推理任务。每次触发输出：检测到高电平→任务开始→分配内存→H2D 拷贝→创建 Stream→执行推理→推理完成→结果摘要。最终获得 3 个推理结果，每个结果形状为 (1, 256)。

In [ ]:
def coordination_demo():
    """GPIO + CANN 协同控制完整演示"""
    print("\n" + "=" * 60)
    print("[模块3] GPIO + CANN 协同控制演示")
    print("=" * 60)

    # 1. 创建GPIO控制器
    gpio = GPIOController(mode=GPIOMode.SIMULATION)
    print("[OK] GPIO控制器创建成功")

    # 2. 创建CANN Runtime
    runtime = CANNRuntime(mode="simulation")
    print("[OK] CANN Runtime创建成功")

    # 3. 创建协同控制器
    coordinator = GPIOCANNCoordinator(
        gpio_controller=gpio,
        runtime=runtime,
        trigger_pin=18
    )
    print("[OK] 协同控制器创建成功")

    # 4. 启动协同控制（仿真模式自动触发3次）
    coordinator.start()

    # 5. 等待完成
    time.sleep(0.5)

    # 6. 获取结果
    results = coordinator.get_results()
    print(f"\n共获得 {len(results)} 个推理结果")

    # 7. 清理
    coordinator.cleanup()
    print("\n[OK] 协同控制演示完成")

    return coordinator

coordinator = coordination_demo()

## 第五部分：实验数据记录与分析

#### 知识点说明

本代码块通过 `analyze_results(coordinator)` 函数对协同控制实验的结果进行多维度分析，培养学生的性能分析能力：

1. **任务执行时间分析**：提取所有推理结果的时间戳，计算相邻任务的时间间隔（平均/最小/最大）。这反映了系统处理连续触发事件的响应能力。在仿真模式下，间隔主要由 `time.sleep` 模拟的推理延迟和触发间隔决定。

2. **推理结果分析**：对每个任务的推理结果计算统计量（形状、均值、标准差）。由于仿真推理使用随机权重矩阵，结果均值应接近 0（ReLU 激活后略大于 0），标准差反映输出分布的离散程度。

3. **性能统计**：从 CANN Runtime 的 `stats` 字典提取总推理次数、总耗时和平均耗时。这些是端侧 AI 部署的关键性能指标（KPI）：
   - **总推理次数**：反映系统吞吐量
   - **平均耗时**：反映单次推理延迟，直接影响用户体验
   - 在真实硬件上，这些指标可用于与 CPU 推理对比，量化 NPU 加速比

4. **实验结论**：验证各模块功能是否正常——GPIO 控制、CANN Runtime 调度、协同流程、数据流管理、多任务并行。每条结论对应一个功能验证点。

#### 预期结果

输出 3 个推理任务的时间间隔统计、每个结果的形状和统计量、性能汇总数据，以及 5 条结论确认各模块功能正常。

In [ ]:
def analyze_results(coordinator):
    """分析实验结果"""
    print("\n" + "=" * 60)
    print("实验结果分析")
    print("=" * 60)

    results = coordinator.get_results()

    if len(results) == 0:
        print("没有结果数据")
        return

    # 1. 任务执行时间分析
    print("\n[1] 任务执行时间分析")
    print("-" * 40)
    times = [r['timestamp'] for r in results]
    if len(times) > 1:
        intervals = [times[i] - times[i-1] for i in range(1, len(times))]
        print(f"  平均间隔: {np.mean(intervals):.2f}秒")
        print(f"  最小间隔: {np.min(intervals):.2f}秒")
        print(f"  最大间隔: {np.max(intervals):.2f}秒")

    # 2. 推理结果分析
    print("\n[2] 推理结果分析")
    print("-" * 40)
    for i, r in enumerate(results):
        result = r['result']
        print(f"  任务{i+1}: 形状{result.shape}, "
              f"均值={result.mean():.4f}, 标准差={result.std():.4f}")

    # 3. 性能统计
    print("\n[3] 性能统计")
    print("-" * 40)
    stats = coordinator.runtime.get_stats()
    print(f"  总推理次数: {stats['inference_calls']}")
    print(f"  总耗时: {stats['total_time']:.2f}秒")
    print(f"  平均耗时: {stats['total_time']/max(1, stats['inference_calls']):.2f}秒")

    # 4. 结论
    print("\n[4] 实验结论")
    print("-" * 40)
    print("[OK] GPIO外设控制功能正常 (仿真/硬件双模式)")
    print("[OK] CANN Runtime资源调度正常 (Stream管理)")
    print("[OK] GPIO触发 -> AI推理 协同流程完整")
    print("[OK] Host-Device数据流管理正常")
    print("[OK] 多任务并行调度验证通过")

analyze_results(coordinator)

## 第六部分：实验总结

#### 知识点说明

本代码块输出实验总结，包含三个模块的完成情况、核心知识点回顾和硬件模式切换指南。这是实验报告的重要组成部分：

1. **实验完成情况**：以表格形式展示三个模块（GPIO 外设控制、CANN Runtime 资源调度、协同控制与部署）的学习成果和关键知识点。

2. **核心知识点**：梳理三大知识领域：
   - GPIO 驱动与 AI 框架联合开发（libgpiod + CANN Runtime 协同调用、硬件抽象层设计）
   - 硬件资源管理与调度（Device/Context/Stream 三层模型、同 Stream 串行/不同 Stream 并行）
   - 端侧部署核心方法（资源初始化与清理、异步执行与流水线、内存管理与数据传输）

3. **实验模式切换指南**：从仿真模式切换到硬件模式的 5 步操作：
   - 安装 gpiod：`sudo apt-get install python3-libgpiod`
   - 确认 CANN 环境已配置
   - 修改 `GPIOController(mode=GPIOMode.HARDWARE)`
   - 修改 `CANNRuntime(mode="hardware")`
   - 以 sudo 运行：`sudo jupyter-notebook --allow-root`

#### 预期结果

输出三个模块的完成情况、核心知识点和从仿真模式切换到硬件模式的 5 步操作指南。

In [ ]:
def experiment_summary():
    """实验总结"""
    print("\n" + "=" * 60)
    print("实验3：GPIO + CANN 协同控制实验 - 总结")
    print("=" * 60)
    print("""
+-------------------------------------------------------------+
|                    实验完成情况                             |
+-------------------------------------------------------------+
|  [完成] 模块1: GPIO外设控制                                 |
|     - 理解libgpiod基本原理                                  |
|     - 掌握GPIO输入/输出操作                                 |
|     - 硬件/仿真双模式实现                                   |
|                                                             |
|  [完成] 模块2: CANN Runtime资源调度                         |
|     - 理解Host-Device异构架构                               |
|     - 掌握Device/Context/Stream概念                         |
|     - 实现H2D/D2H数据传输                                   |
|                                                             |
|  [完成] 模块3: 协同控制与部署                               |
|     - GPIO触发 -> AI推理完整流程                            |
|     - 多Stream并行调度                                      |
|     - 端侧部署模式理解                                      |
+-------------------------------------------------------------+

+-------------------------------------------------------------+
|                    核心知识点                               |
+-------------------------------------------------------------+
|  1. GPIO驱动与AI框架联合开发                                |
|     - libgpiod + CANN Runtime协同调用                       |
|     - 硬件抽象层设计 (HAL)                                  |
|                                                             |
|  2. 硬件资源管理与调度                                      |
|     - Device/Context/Stream三层模型                         |
|     - 同Stream串行，不同Stream并行                          |
|                                                             |
|  3. 端侧部署核心方法                                        |
|     - 资源初始化与清理                                      |
|     - 异步执行与流水线                                      |
|     - 内存管理与数据传输                                    |
+-------------------------------------------------------------+

+-------------------------------------------------------------+
|                    实验模式切换                             |
+-------------------------------------------------------------+
|  当前模式: 仿真模式 (无需硬件)                              |
|                                                             |
|  切换到硬件模式:                                            |
|  1. 安装gpiod: sudo apt-get install python3-libgpiod        |
|  2. 确认CANN环境已配置                                      |
|  3. 修改 GPIOController(mode=GPIOMode.HARDWARE)             |
|  4. 修改 CANNRuntime(mode="hardware")                       |
|  5. 以sudo运行: sudo jupyter-notebook --allow-root          |
+-------------------------------------------------------------+
""")

experiment_summary()

## 附录：完整实验代码（整合为一个 Cell）

#### 知识点说明

如果想快速运行完整实验，可以使用以下整合代码。该代码将 `GPIOController`、`CANNRuntime` 和 `GPIOCANNCoordinator` 三个类以及完整实验执行流程整合在一个 Cell 中，可直接运行。

**整合代码与分步代码的关系**：逻辑与上方分步代码完全一致，但做了适当精简（如移除部分异常处理和注释），适合快速验证或一键运行完整实验。在实际开发中，推荐使用分步方式以便理解和调试。

**适用场景**：
- 快速验证实验环境是否正常
- 一键演示完整实验流程
- 作为独立脚本在非 Jupyter 环境中运行

#### 预期结果

输出实验标题→GPIO 仿真模式启用→CANN 仿真模式初始化→协同控制器初始化→启动协同控制→3 次自动触发及推理→获得 3 个推理结果（含形状和均值）→资源清理→实验完成。

In [ ]:
# 实验3完整代码（可直接在Jupyter中运行）

import os
import sys
import time
import threading
import numpy as np
from enum import Enum
from dataclasses import dataclass
from typing import Optional, List, Tuple, Callable
import warnings
warnings.filterwarnings('ignore')

# ==================== 第一部分：GPIO控制器 ====================

class GPIOMode(Enum):
    SIMULATION = "simulation"
    HARDWARE = "hardware"

@dataclass
class GPIOEvent:
    pin: int
    value: int
    timestamp: float
    description: str

class GPIOController:
    def __init__(self, mode: GPIOMode = GPIOMode.SIMULATION):
        self.mode = mode
        self._pins = {}
        self._events = []
        self._callbacks = {}
        if mode == GPIOMode.HARDWARE:
            self._init_hardware()
        else:
            self._init_simulation()

    def _init_simulation(self):
        print("[GPIO] 仿真模式已启用")
        for pin in [17, 18, 22, 23, 24, 25]:
            self._pins[pin] = 0

    def _init_hardware(self):
        try:
            import gpiod
            self._gpiod = gpiod
            self._chip = gpiod.Chip('gpiochip0')
            print("[GPIO] 硬件模式初始化成功")
        except Exception:
            print("[GPIO] 硬件初始化失败，切换到仿真模式")
            self.mode = GPIOMode.SIMULATION
            self._init_simulation()

    def write(self, pin: int, value: int):
        if self.mode == GPIOMode.HARDWARE:
            try:
                line = self._chip.get_line(pin)
                line.request(consumer="gpio_control",
                             type=self._gpiod.LINE_REQ_DIR_OUT)
                line.set_value(value)
                line.release()
            except Exception:
                return False
        else:
            self._pins[pin] = value
            print(f"[GPIO仿真] GPIO{pin} = {value}")
        self._events.append(GPIOEvent(pin=pin, value=value,
                                      timestamp=time.time(),
                                      description=f"Write to GPIO{pin}"))
        return True

    def read(self, pin: int) -> int:
        if self.mode == GPIOMode.HARDWARE:
            try:
                line = self._chip.get_line(pin)
                line.request(consumer="gpio_control",
                             type=self._gpiod.LINE_REQ_DIR_IN)
                value = line.get_value()
                line.release()
                return value
            except Exception:
                return 0
        else:
            return self._pins.get(pin, 0)

    def on_event(self, pin: int, callback: Callable):
        if pin not in self._callbacks:
            self._callbacks[pin] = []
        self._callbacks[pin].append(callback)
        return self

    def _trigger_callbacks(self, pin: int, value: int):
        if pin in self._callbacks:
            for cb in self._callbacks[pin]:
                try:
                    cb(pin, value)
                except Exception:
                    pass

    def simulate_trigger(self, pin: int, duration: float = 0.5):
        if self.mode != GPIOMode.SIMULATION:
            return
        print(f"[GPIO仿真] 模拟触发 GPIO{pin} (持续{duration}s)")
        self.write(pin, 1)
        self._trigger_callbacks(pin, 1)
        time.sleep(duration)
        self.write(pin, 0)
        self._trigger_callbacks(pin, 0)

    def get_events(self) -> List[GPIOEvent]:
        return self._events

    def cleanup(self):
        print("[GPIO] 资源清理完成")


# ==================== 第二部分：CANN Runtime ====================

class CANNRuntime:
    def __init__(self, device_id: int = 0, mode: str = "auto"):
        self.device_id = device_id
        self.mode = mode if mode != "auto" else "simulation"
        self.streams = []
        self.device_memory = []
        self.stats = {'h2d_transfers': 0, 'd2h_transfers': 0,
                      'inference_calls': 0, 'total_time': 0}
        print(f"[CANN] Runtime模式: {self.mode}")

    def create_stream(self) -> str:
        stream_id = f"stream_{len(self.streams)}"
        self.streams.append(stream_id)
        print(f"[CANN] 创建Stream: {stream_id}")
        return stream_id

    def allocate_memory(self, size: int) -> int:
        mem_addr = id(self) + len(self.device_memory)
        self.device_memory.append(mem_addr)
        print(f"[CANN] 分配内存: {size}字节 -> 地址{hex(mem_addr)}")
        return mem_addr

    def memcpy_h2d(self, host_data: np.ndarray, dev_mem: int):
        self.stats['h2d_transfers'] += 1
        print(f"[CANN] H2D拷贝: {host_data.nbytes}字节")
        self._last_h2d_data = host_data.copy()

    def memcpy_d2h(self, dev_mem: int, size: int) -> np.ndarray:
        self.stats['d2h_transfers'] += 1
        print(f"[CANN] D2H拷贝: {size}字节")
        return np.random.randn(size // 4).astype(np.float32)

    def execute_inference(self, input_data: np.ndarray, stream=None) -> np.ndarray:
        self.stats['inference_calls'] += 1
        start = time.time()
        print(f"[推理] 开始执行 (输入形状: {input_data.shape})")
        data = input_data.reshape(1, -1) if input_data.ndim == 1 else input_data
        weight = np.random.randn(data.shape[-1], data.shape[-1]) * 0.1
        output = np.maximum(data @ weight, 0)
        time.sleep(min(0.05 + data.size / 100000, 0.3))
        self.stats['total_time'] += time.time() - start
        print(f"[推理] 完成，耗时: {time.time()-start:.4f}秒")
        return output.astype(np.float32)

    def get_stats(self) -> dict:
        return self.stats

    def cleanup(self):
        print("[CANN] 资源清理完成")


# ==================== 第三部分：协同控制器 ====================

class GPIOCANNCoordinator:
    def __init__(self, gpio: GPIOController, runtime: CANNRuntime, trigger_pin: int = 18):
        self.gpio = gpio
        self.runtime = runtime
        self.trigger_pin = trigger_pin
        self.running = False
        self.task_counter = 0
        self.results = []
        self.gpio.on_event(trigger_pin, self._on_gpio_trigger)
        print(f"[协调器] 初始化完成，触发引脚: GPIO{trigger_pin}")

    def _on_gpio_trigger(self, pin: int, value: int):
        if value == 1 and self.running:
            self.task_counter += 1
            input_data = np.random.randn(1, 256).astype(np.float32)
            print(f"\n[触发] 任务{self.task_counter} 开始处理")
            stream = self.runtime.create_stream()
            result = self.runtime.execute_inference(input_data, stream)
            self.results.append({'id': self.task_counter, 'result': result,
                                 'timestamp': time.time()})
            print(f"[任务 {self.task_counter}] 完成")

    def start(self):
        print("\n" + "=" * 60)
        print("[协调器] 启动协同控制系统")
        print("=" * 60)
        self.running = True
        if self.gpio.mode == GPIOMode.SIMULATION:
            print("[仿真] 自动模拟触发信号...")
            for i in range(3):
                self.gpio.simulate_trigger(self.trigger_pin, duration=0.5)
                time.sleep(1.2)
            self.stop()

    def stop(self):
        self.running = False
        print(f"\n[协调器] 完成 {len(self.results)} 个任务")

    def get_results(self) -> List[dict]:
        return self.results

    def cleanup(self):
        self.gpio.cleanup()
        self.runtime.cleanup()
        print("[协调器] 资源清理完成")


# ==================== 第四部分：执行实验 ====================

print("=" * 60)
print("实验3：GPIO + CANN 协同控制实验")
print("=" * 60)

gpio = GPIOController(mode=GPIOMode.SIMULATION)
runtime = CANNRuntime(mode="simulation")
coordinator = GPIOCANNCoordinator(gpio, runtime, trigger_pin=18)

coordinator.start()

results = coordinator.get_results()
print(f"\n获得 {len(results)} 个推理结果:")
for r in results:
    print(f"  任务{r['id']}: 结果形状{r['result'].shape}, 均值={r['result'].mean():.4f}")

coordinator.cleanup()

print("\n" + "=" * 60)
print("[OK] 实验完成！")
print("=" * 60)

## 扩展实验

在完成本实验主体内容后，建议学习者尝试以下扩展实验。这些实验以初学者为主，简单可行，均可在仿真模式下完成，无需任何硬件。每个扩展实验只需提出要求，学习者可自行尝试实现，参考答案位于 `answer/` 目录下。

> **学习建议**：先独立思考并尝试实现，遇到困难时再参考 `answer/` 目录下的参考实现。每个扩展实验的参考代码均可独立运行。

---

### 扩展实验1：GPIO 按键防抖（软件去抖动）

**实验要求**：

在真实硬件中，按键按下和释放瞬间会产生机械抖动，导致 GPIO 电平在短时间内反复跳变。请基于本实验的 `GPIOController`，实现一个**软件防抖函数**：

1. 编写 `debounced_read(gpio, pin, stable_count=3, interval=0.05)` 函数
2. 连续 `stable_count` 次读取到相同电平值才确认状态变化，每次读取间隔 `interval` 秒
3. 返回稳定后的电平值
4. 在仿真模式下测试：对某引脚写入 1，调用防抖读取，验证返回值为 1；再写入 0，验证返回值为 0

**参考答案**：`answer/extension1_debounce.py`

---

### 扩展实验2：软件 PWM 模拟 LED 呼吸灯

**实验要求**：

PWM（脉宽调制）通过快速切换 GPIO 高低电平来模拟中间亮度。请实现一个简单的**软件 PWM 呼吸灯**效果：

1. 编写 `software_pwm(gpio, pin, duty_cycle, duration, freq=100)` 函数，`duty_cycle` 为占空比（0.0~1.0）
2. 编写 `breathing_led(gpio, pin, cycles=3, steps=20)` 函数，实现 LED 渐亮渐暗的呼吸效果
3. 在一个呼吸周期内，占空比从 0 渐变到 1 再回到 0，共 `steps` 步
4. 在仿真模式下运行 3 个呼吸周期，观察 GPIO 状态变化输出

**参考答案**：`answer/extension2_pwm_breathing.py`

---

### 扩展实验3：多 GPIO 引脚批量操作与状态显示

**实验要求**：

实际应用中常需要同时控制多个 GPIO（如 LED 阵列）。请扩展 `GPIOController` 的功能：

1. 编写 `batch_write(gpio, pin_values: dict)` 函数，一次性设置多个引脚的电平，如 `{17: 1, 18: 0, 22: 1}`
2. 编写 `led_bar_display(gpio, pins, values)` 函数，用文本字符绘制 LED 状态条，如 `[█ ▄ █ ▄ █]` 表示 5 个 LED 的亮灭状态（█=亮，▄=灭）
3. 模拟 8 个 LED 的跑马灯效果：依次点亮每个 LED，同时熄灭其他，循环 2 轮
4. 每步打印 LED 状态条，直观展示跑马灯效果

**参考答案**：`answer/extension3_batch_led.py`

---

### 扩展实验4：推理结果阈值判断控制 GPIO 输出

**实验要求**：

在端侧 AI 部署中，常需要根据推理结果控制外设（如报警指示灯）。请实现**推理结果驱动的 GPIO 控制**：

1. 执行一次 AI 推理（使用 `CANNRuntime.execute_inference`），获取推理结果
2. 计算推理结果的均值作为判断指标
3. 设置阈值 `threshold`（如 0.0），当均值超过阈值时点亮 GPIO23（报警指示灯），低于阈值时熄灭
4. 循环执行 5 次推理，每次根据结果控制 GPIO23，并打印推理均值和 LED 状态
5. 统计 5 次中报警触发次数

**参考答案**：`answer/extension4_threshold_alarm.py`

---

### 扩展实验5：GPIO 事件时间线记录与可视化

**实验要求**：

事件时间线可视化是调试嵌入式系统的重要手段。请实现 GPIO 事件的**时间线记录与文本可视化**：

1. 创建 GPIOController，执行一系列操作：对 GPIO17/18/22/23 各写入若干次高低电平，操作间加入随机间隔
2. 获取所有事件记录（`get_events()`）
3. 编写 `draw_timeline(events, width=60)` 函数，用文本字符绘制事件时间线：
   - 横轴为时间，纵轴为各引脚
   - 用 `━` 表示高电平持续段，`─` 表示低电平持续段，`┃` 表示跳变时刻
4. 打印时间线图和事件统计表（每个引脚的操作次数、高电平总持续时间）

**参考答案**：`answer/extension5_event_timeline.py`

---

> **扩展实验提示**：以上 5 个扩展实验难度递增，建议按顺序完成。所有实验均可在仿真模式下运行，无需昇腾硬件。完成后可尝试将代码切换到硬件模式，在真实开发板上验证。

## 实验完成标志

- [ ] 理解昇腾软件体系四层架构及 CANN 异构计算原理
- [ ] 理解嵌入式 Linux 启动流程与设备树机制
- [ ] 理解 Linux 驱动三种开发方式及 libgpiod 用户空间 GPIO 控制
- [ ] GPIO 控制器正常工作（仿真模式）
- [ ] CANN Runtime 资源调度正常
- [ ] GPIO 触发→AI 推理协同流程完整
- [ ] 多任务并行调度验证通过
- [ ] 理解端侧部署核心方法
- [ ] 完成至少 2 个扩展实验
- [ ] 可切换到硬件模式运行（可选）

## 参考资料

- 昇腾 CANN 官方文档：https://www.hiascend.com/document
- libgpiod 文档与源码：https://git.kernel.org/pub/scm/libs/libgpiod/libgpiod.git/
- ACL Runtime API 参考手册
- 昇腾社区技术文章与开发者论坛
- 《深入浅出 Linux 设备树》——AIpro 外设驱动加载机制
- Linux 内核 GPIO 子系统文档：Documentation/gpio/